### TODO 4

Create your model.

In [ ]:
# import torch
# import torch.nn as nn
# # from huggingface_hub import HfApi, hf_hub_download


# class SimpleVideoClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # convolution block on centroids array
#         self.conv = nn.Sequential(
            
#             nn.Conv1d(2, 16, kernel_size = 5, padding = 2),
#             nn.ReLU(),
#             nn.MaxPool1d(kernel_size = 2), # 100 frames --> 50 frames

#             nn.Conv1d(16, 32, kernel_size = 3, padding = 1),
#             nn.ReLU(),
#             nn.AdaptiveAvgPool1d(1) # 50 frames --> 1 "frame"
#         )

#         # Linear layers
#         self.fc1 = nn.Linear(32, 1)

#     def forward(self, x):
#         # rearrange dimensions (B, T, C, H, W)
#         x_perm = x.permute(0, 2, 1, 3, 4)
#         B, T, C, H, W = x_perm.shape

#         # convert to centroids (B x T x 2)
#         h1 = torch.stack([get_trunk_centroids(v) for v in x_perm])
#         h1 = h1.to(torch.float32)

#         # normalizer = torch.tensor([W, H], device=h1.device, dtype=torch.float32)
#         # h1_norm = h1 / normalizer

#         h1_t = h1.transpose(1, 2)
#         h2 = self.conv(h1_t)
#         h2_flat = torch.flatten(h2, 1)
#         h3 = self.fc1(h2_flat)
#         # h4 = self.fc2(h3)

#         return h3


# First Attempt, no centroids calculation

**Update**, vanishing gradients, consistently just predicts the mean

In [ ]:
# class VideoConvModel(nn.Module):
#     def __init__(self):
#         super().__init__()
        
#         # Spatial Features
#         # We can use a tiny custom CNN or a pre-trained ResNet18
#         self.spatial_features = nn.Sequential(
#             nn.Conv2d(3, 6, kernel_size=3, stride=2, padding=1), # 3x224x224 --> 6x112x112
#             nn.ReLU(),
#             # nn.MaxPool2d(2),
#             nn.Conv2d(6, 12, kernel_size=3, stride=2, padding=1), #6x112x112 --> 12x56x56
#             nn.ReLU(),
#             # nn.MaxPool2d(2),
#             nn.Conv2d(12, 18, kernel_size=3, stride=2, padding=1), #12x56x56 --> 18x28x28
#             nn.ReLU(),
#             nn.Conv2d(18, 24, kernel_size=3, stride=2, padding=1), #18x28x28 --> 24x14x14
#             nn.ReLU(),
#             nn.Conv2d(24, 32, kernel_size=3, stride=2, padding=1), #24x14x14 --> 32x7x7
#             nn.ReLU(),
#             nn.Conv2d(32, 40, kernel_size=3, stride=2, padding=1), #32x7x7 --> 40x4x4
#             nn.ReLU(),
#             nn.Flatten(), # Becomes 40 * 4 * 4 = 640
#             nn.Linear(640, 2) # 640x1x1 -> 2x1x1
#             # nn.ReLU()
#         )

#         # Temporal Processor
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(2, 6, kernel_size = 3, stride = 2, padding = 1), # 100x2x1x1 frames --> 50x6x1x1 frames
#             nn.ReLU(),
#             nn.Conv1d(6, 12, kernel_size = 3, stride = 2, padding = 1), # 50x6x1x1 frames --> 25x12x1x1 frames
#             nn.ReLU(),
#             nn.Conv1d(12, 18, kernel_size = 3, stride = 2, padding = 1), # 25x12x1x1 frames --> 13x18x1x1 frames
#             nn.ReLU()
#         )
        
#         # classifying summary of spatial/temporal info
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(234, 1) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # We want [B*T, C, H, W] for the spatial CNN
#         # Transpose B and C to get [T, B, C, H, W] then flatten T and B
#         x_in = x.transpose(0, 2).reshape(T * B, C, H, W) 
#         features = self.spatial_features(x_in) # [B*T, 32, 1, 1]
        
#         # Reshape back to separate Time and Batch
#         features = features.reshape(B, T, -1).transpose(1, 2) # [B, 2, T]
        
#         # Run temporal convolutions
#         combined = self.temporal_conv(features).squeeze(-1) # [B, 64]
#         return self.classifier(combined)

# Second attempt, healthier model
Our model is just predicting the mean of each batch, which signals a vanishing gradients problem
Let's help the model pass more gradients by:
- Batch Normalization
- removing the linear bottleneck at the end of 
- increasing number of channels early in the spatial convolutions

**UPDATE** results in bad gradients (likely vanishing) consistently just predicts the mean

In [ ]:
# class VideoConvModel(nn.Module):
#     def __init__(self):
#         super().__init__()
        
#         # Spatial Features
#         # We can use a tiny custom CNN or a pre-trained ResNet18
#         self.spatial_features = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), # 3x224x224 --> 6x112x112
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), #6x112x112 --> 12x56x56
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), #12x56x56 --> 18x28x28
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), #18x28x28 --> 24x14x14
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1), #24x14x14 --> 32x7x7
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1), #32x7x7 --> 40x4x4
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Flatten(), # Becomes 128 * 4 * 4 = 640
#             nn.Linear(128 * 4 * 4, 64) # largex1x1 -> 64x1x1
#             # nn.ReLU()
#         )

#         # Temporal Processor
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(64, 64, kernel_size = 3, stride = 2, padding = 1), # 100x64x1x1 frames --> 50x64x1x1 frames
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.Conv1d(64, 128, kernel_size = 3, stride = 2, padding = 1), # 50x64x1x1 frames --> 25x128x1x1 frames
#             nn.BatchNorm1d(128),
#             nn.ReLU(),
#             nn.Conv1d(128, 128, kernel_size = 3, stride = 2, padding = 1), # 25x128x1x1 frames --> 13x128x1x1 frames
#             nn.BatchNorm1d(128),
#             nn.ReLU()
#         )
        
#         # classifying summary of spatial/temporal info
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(1664, 1) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # We want [B*T, C, H, W] for the spatial CNN
#         # Transpose B and C to get [T, B, C, H, W] then flatten T and B
#         x_in = x.transpose(0, 2).reshape(T * B, C, H, W) 
#         features = self.spatial_features(x_in) # [B*T, 32, 1, 1]
        
#         # Reshape back to separate Time and Batch
#         features = features.reshape(B, T, -1).transpose(1, 2) # [B, 2, T]
        
#         # Run temporal convolutions
#         combined = self.temporal_conv(features).squeeze(-1) # [B, 64]
#         return self.classifier(combined)

# Third attempt, removing another bottleneck:
- The final linear classifier bottleneck scares me, trying to make it less aggressive.
- Also, three frames seems too fine-grained of a temporal kernel to actually see any push-up movement, gonna expand it to 5 and make stride larger

**Update** still have a gradients problem - very often just predict the mean of the batch

In [ ]:
# class VideoConvModel(nn.Module):
#     def __init__(self):
#         super().__init__()
        
#         # Spatial Features
#         # We can use a tiny custom CNN or a pre-trained ResNet18
#         self.spatial_features = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1), # 3x224x224 --> 6x112x112
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), #6x112x112 --> 12x56x56
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), #12x56x56 --> 18x28x28
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), #18x28x28 --> 24x14x14
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1), #24x14x14 --> 32x7x7
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1), #32x7x7 --> 40x4x4
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.Flatten(), # Becomes 128 * 4 * 4 = 640
#             nn.Linear(128 * 4 * 4, 64), # largex1x1 -> 64x1x1
#             nn.ReLU()
#         )

#         # Temporal Processor
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(64, 64, kernel_size = 5, stride = 3, padding = 2), # 100x64x1x1 frames --> 33x64x1x1 frames
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.Conv1d(64, 128, kernel_size = 5, stride = 3, padding = 2), # 33x64x1x1 frames --> 11x128x1x1 frames
#             nn.BatchNorm1d(128),
#             nn.ReLU(),
#             nn.Conv1d(128, 128, kernel_size = 5, stride = 3, padding = 2), # 11x128x1x1 frames --> 4x128x1x1 frames
#             nn.BatchNorm1d(128),
#             nn.ReLU()
#         )
        
#         # classifying summary of spatial/temporal info
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(512, 1) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # We want [B*T, C, H, W] for the spatial CNN
#         # Transpose B and C to get [T, B, C, H, W] then flatten T and B
#         x_in = x.transpose(0, 2).reshape(T * B, C, H, W) 
#         features = self.spatial_features(x_in) # [B*T, 32, 1, 1]
        
#         # Reshape back to separate Time and Batch
#         features = features.reshape(B, T, -1).transpose(1, 2) # [B, 2, T]
        
#         # Run temporal convolutions
#         combined = self.temporal_conv(features) # [B, 64]
#         return self.classifier(combined)

# Fourth attempt - switching to Resnet
- trying to solve the vanishing gradients problem

**Update**, does well on training, not so much on test (overfitting)

In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoResNetModel(nn.Module):
#     def __init__(self, use_pretrained=True):
#         super().__init__()
        
#         # 1. Load Pre-trained ResNet18
#         # weights=models.ResNet18_Weights.DEFAULT is the modern way to load pre-trained
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
        
#         # 2. Extract the feature extractor (all layers except the final fc layer)
#         # ResNet18's final layer before the classifier is an Global Average Pool
#         # which outputs 512 features.
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored
        
#         # 3. Optional: Freeze the backbone if you have a small dataset
#         # for param in self.backbone.parameters():
#         #     param.requires_grad = False

#         # 4. Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(512, 128, kernel_size=3, stride=2, padding=1), # 100 -> 50
#             nn.BatchNorm1d(128),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 50 -> 25
#             nn.BatchNorm1d(64),
#             nn.LeakyReLU(0.1),
#             nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 25 -> 13
#             nn.BatchNorm1d(32),
#             nn.LeakyReLU(0.1)
#         )
        
#         # 5. Classifier
#         # Output of temporal is [B, 32, 13]. Flattened = 32 * 13 = 416
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(416, 1) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # Reshape for Spatial: [B*T, C, H, W]
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
#         # Pass through ResNet backbone
#         # Output is [B*T, 512, 1, 1]
#         spatial_features = self.backbone(x_in) 
        
#         # Reshape for Temporal: [B, 512, T]
#         temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
#         # Temporal Conv
#         temporal_out = self.temporal_conv(temporal_in)
        
#         # Classify
#         return self.classifier(temporal_out)

# Fifth attempt (coming back to this)

adding transformations to the input data so our training dataset grows

In [ ]:
# # Redefine dataloaders function to include transformations
# def get_dataloaders(video_dir, batch_size = 5, val_split = 0.2, collate_fn = None):
#     """Create train and validation dataloaders."""

#     # applying train_transform to the full dataset is naive I think
#     full_dataset = VideoDataset(video_dir, transform = train_transform)

#     val_size = int(len(full_dataset) * val_split)
#     train_size = len(full_dataset) - val_size

#     train_dataset, val_dataset = random_split(
#         full_dataset,
#         [train_size, val_size],
#         generator=torch.Generator().manual_seed(42)
#     )

#     train_loader = DataLoader(
#         train_dataset,
#         batch_size=batch_size,
#         shuffle=True,
#         num_workers=0,
#         collate_fn=collate_fn
#     )

#     val_loader = DataLoader(
#         val_dataset,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=0,
#         collate_fn=collate_fn
#     )

#     print(f"Train: {len(train_dataset)} videos, Val: {len(val_dataset)} videos\n")

#     return train_loader, val_loader

# Fifth attempt: preventing overfitting
- adding weight decay in the optimizer
- unfreezing last layer of resnet backbone

In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoResNetModel(nn.Module):
#     def __init__(self, use_pretrained=True):
#         super().__init__()
        
#         # 1. Load Pre-trained ResNet18
#         # weights=models.ResNet18_Weights.DEFAULT is the modern way to load pre-trained
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
        
#         # 2. Extract the feature extractor (all layers except the final fc layer)
#         # ResNet18's final layer before the classifier is an Global Average Pool
#         # which outputs 512 features.
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored

#         # for name, child in self.backbone.named_children():
#         #     if name == '7': # In a Sequential(resnet_children[:-1]), layer4 is index 7
#         #         print("Unfreezing Layer 4...")
#         #         for param in child.parameters():
#         #             param.requires_grad = True
                
#         # 3. Optional: Freeze the backbone if you have a small dataset
#         # for param in self.backbone.parameters():
#         #     param.requires_grad = False

#         # 4. Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(512, 128, kernel_size=3, stride=2, padding=1), # 100 -> 50
#             nn.BatchNorm1d(128),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 50 -> 25
#             nn.BatchNorm1d(64),
#             nn.LeakyReLU(0.1),
#             nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 25 -> 13
#             nn.BatchNorm1d(32),
#             nn.LeakyReLU(0.1)
#         )
        
#         # 5. Classifier
#         # Output of temporal is [B, 32, 13]. Flattened = 32 * 13 = 416
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(416, 1) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # Reshape for Spatial: [B*T, C, H, W]
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
#         # Pass through ResNet backbone
#         # Output is [B*T, 512, 1, 1]
#         spatial_features = self.backbone(x_in) 
        
#         # Reshape for Temporal: [B, 512, T]
#         temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
#         # Temporal Conv
#         temporal_out = self.temporal_conv(temporal_in)
        
#         # Classify
#         return self.classifier(temporal_out)

# Sixth Attempt: Trying transformations on the data
- implement base transformations
- wrap transformation implementation in a class which guarantees transforms are applied uniformly across each frame in a time-series
- lowering learning rate, and slightly raising weight decay
- monitoring weight norms, to see if weights explode or vanish
- training for MANY more epochs (60), and saving the best model across all epochs to use later

In [ ]:
import random
import torchvision.transforms as T

# ## transformations (Copied from Practical 9)
# train_transform = T.Compose([
    
#     T.Resize( (224, 224) ),
#     T.RandomCrop(150), 
    
#     T.RandomHorizontalFlip(p = .1),
#     T.RandomVerticalFlip(p = .1),
#     T.RandomRotation(degrees = 10), 
    
#     T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    
#     # T.ToTensor(),

#     T.Normalize(mean = [ 0.485, 0.456, 0.406 ],std = [ 0.229, 0.224, 0.225 ])
# ])

# val_transform = T.Compose([
    
#     T.Resize( (224, 224) ),
#     T.CenterCrop(150), 
#     # T.ToTensor(),
#     T.Normalize(mean = [ 0.485, 0.456, 0.406 ],std = [ 0.229, 0.224, 0.225 ]) 
# ])

# class to ensure transformations are applied uniformly across the time axis
class TemporalUniformTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, video_frames):
        """
        video_frames: A list of PIL Images or a 4D Tensor [T, C, H, W]
        """
        # 1. Generate a single random seed for this entire video clip
        seed = random.randint(0, 2**32)
        
        video_frames = torch.unbind(video_frames, dim = 1)

        transformed_frames = []
        
        for frame in video_frames:
            # 2. Synchronize all random engines to the same seed for this frame
            random.seed(seed)
            torch.manual_seed(seed)
            np.random.seed(seed)
            
            # 3. Apply the transform (it will now pick the same 'random' values)
            transformed_frames.append(self.transform(frame))
            
        # 4. Stack back into a 4D tensor [C, T, H, W] for the model
        return torch.stack(transformed_frames, dim=1)


# new class so a different transform can be applied to each dataset (train vs val)
class ApplyTransform(Dataset):
    """A small wrapper to apply a specific transform to a dataset subset."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

# redefine dataloaders with transforms correctly appleid to train and val
def get_dataloaders(video_dir, batch_size=5, val_split=0.2, train_transform=None, val_transform=None, collate_fn=None):
    
    # 1. Initialize dataset WITHOUT a transform initially
    full_dataset = VideoDataset(video_dir, transform=None)

    val_size = int(len(full_dataset) * val_split)
    train_size = len(full_dataset) - val_size

    # 2. Split the raw data
    train_subset, val_subset = random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    # 3. Wrap subsets with their respective transforms
    train_dataset = ApplyTransform(train_subset, transform=train_transform)
    val_dataset = ApplyTransform(val_subset, transform=val_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
    )

    return train_loader, val_loader


# Seventh attempt
- the original transformations were likely too difficult. Let's try easier ones

In [ ]:
# train_transform = T.Compose([
    
#     T.Resize((240, 240)), 
#     T.RandomCrop(224), 
    
#     T.RandomHorizontalFlip(p=0.5), 
#     T.RandomRotation(degrees=5), # Reduced from 10 to 5
    
#     T.ColorJitter(brightness=0.05, contrast=0.05),
    
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # VALIDATION: Deterministic & Clean
# val_transform = T.Compose([
    
#     T.Resize((224, 224)), 
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
# ])

# Eighth attempt
- let's remove the video transformations, since they seem to be making things worse
- let's change the problem to classification and see if the model has an easier time

In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoResNetModel(nn.Module):
#     def __init__(self, use_pretrained=True, num_classes = 10):
#         super().__init__()
        
#         # 1. Load Pre-trained ResNet18
#         # weights=models.ResNet18_Weights.DEFAULT is the modern way to load pre-trained
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
        
#         # 2. Extract the feature extractor (all layers except the final fc layer)
#         # ResNet18's final layer before the classifier is an Global Average Pool
#         # which outputs 512 features.
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored

#         # for name, child in self.backbone.named_children():
#         #     if name == '7': # In a Sequential(resnet_children[:-1]), layer4 is index 7
#         #         print("Unfreezing Layer 4...")
#         #         for param in child.parameters():
#         #             param.requires_grad = True
                
#         # 3. Optional: Freeze the backbone if you have a small dataset
#         # for param in self.backbone.parameters():
#         #     param.requires_grad = False

#         # 4. Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(512, 128, kernel_size=3, stride=2, padding=1), # 100 -> 50
#             nn.BatchNorm1d(128),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 50 -> 25
#             nn.BatchNorm1d(64),
#             nn.LeakyReLU(0.1),
#             nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 25 -> 13
#             nn.BatchNorm1d(32),
#             nn.LeakyReLU(0.1)
#         )
        
#         # 5. Classifier
#         # Output of temporal is [B, 32, 13]. Flattened = 32 * 13 = 416
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(416, num_classes) 
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # Reshape for Spatial: [B*T, C, H, W]
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
#         # Pass through ResNet backbone
#         # Output is [B*T, 512, 1, 1]
#         spatial_features = self.backbone(x_in) 
        
#         # Reshape for Temporal: [B, 512, T]
#         temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
#         # Temporal Conv
#         temporal_out = self.temporal_conv(temporal_in)
        
#         # Classify
#         return self.classifier(temporal_out)

# Ninth attempt:
- previous attempt was best so far (61% final val error)
- replace 1D convs with transformer arch for temporal processing

In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoTransformerModel(nn.Module):
#     def __init__(self, use_pretrained=True, num_classes = 10, num_frames = 100):
#         super().__init__()
        
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored

    
#         # Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         self.d_model = 512  # ResNet18 output size
#         self.nhead = 4      # Number of attention heads
#         self.num_layers = 2 # Number of transformer layers
        
#         # 3. Positional Encoding & CLS Token
#         # We need a learnable position embedding for the 100 frames + 1 for the CLS token
#         self.pos_embedding = nn.Parameter(torch.zeros(1, num_frames + 1, self.d_model))
#         self.cls_token = nn.Parameter(torch.zeros(1, 1, self.d_model))
        
#         # 4. Temporal Transformer Encoder
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model = self.d_model, 
#             nhead = self.nhead,
#             dim_feedforward = 1024,
#             dropout = 0.1,
#             batch_first = True # This makes input shape (Batch, Seq, Features)
#         )
#         self.transformer = nn.TransformerEncoder(encoder_layer, num_layers = self.num_layers)
        
        
#         self.classifier = nn.Linear(self.d_model, num_classes)

#     def forward(self, x):
        
#         B, C, T, H, W = x.shape
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W)    # Reshape for Spatial: [B*T, C, H, W]
#         spatial_features = self.backbone(x_in)              # Output is [B*T, 512, 1, 1]
        
#         # Reshape for Temporal: [B, 512, T]
#         temporal_in = spatial_features.view(B, T, self.d_model)
        
#         # Prepend CLS token to the sequence
#         # cls_tokens shape: [B, 1, 512]
#         cls_tokens = self.cls_token.expand(B, -1, -1)
#         x = torch.cat((cls_tokens, temporal_in), dim=1) # [B, T+1, 512]
        
#         # Add Positional Encoding
#         x = x + self.pos_embedding
        
#         # Transformer Pass
#         # x shape: [B, T+1, 512]
#         transformer_out = self.transformer(x)
        
#         # We take only the first token (the CLS token) which now contains 
#         # the "summary" of the whole temporal sequence
#         sequence_summary = transformer_out[:, 0] # [B, 512]
        
#         return self.classifier(sequence_summary)

# Tenth attempt
- going back to pre-transformer architecture, 
- more layers to increase capacity
- training for longer, with some weight decay to combat overfitting


In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoResNetModel(nn.Module):
#     def __init__(self, use_pretrained=True, num_classes = 10):
#         super().__init__()
        
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored

#         # Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         # added one more convolutional layer than before
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(512, 256, kernel_size=3, stride=2, padding=1), # 100 -> 50
#             nn.BatchNorm1d(256),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Conv1d(256, 128, kernel_size=3, stride=2, padding=1), # 50 -> 25
#             nn.BatchNorm1d(128),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 25 -> 13
#             nn.BatchNorm1d(64),
#             nn.LeakyReLU(0.1),
#             nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 13 -> 7
#             nn.BatchNorm1d(32),
#             nn.LeakyReLU(0.1)
#         )
        
#         # 5. Classifier
#         # Output of temporal is [B, 32, 13]. Flattened = 32 * 7 = 224
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(224, 112),
#             nn.LeakyReLU(0.1),
#             nn.Linear(112, num_classes),
#         )

#     def forward(self, x):
#         # x: [B, C, T, H, W]
#         B, C, T, H, W = x.shape
        
#         # Reshape for Spatial: [B*T, C, H, W]
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
#         # Pass through ResNet backbone
#         # Output is [B*T, 512, 1, 1]
#         spatial_features = self.backbone(x_in) 
        
#         # Reshape for Temporal: [B, 512, T]
#         temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
#         # Temporal Conv
#         temporal_out = self.temporal_conv(temporal_in)
        
#         # Classify
#         return self.classifier(temporal_out)

# Lighter transformations

In [ ]:
# # more tepid transformations now (no horizontal flip, more color jitter)
# train_transform = T.Compose([
    
#     T.Resize((240, 240)), 
#     T.RandomCrop(224), 
    
#     T.RandomRotation(degrees=5), # Reduced from 10 to 5
    
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation = .1),
    
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # VALIDATION: Deterministic & Clean
# val_transform = T.Compose([
    
#     T.Resize((224, 224)), 
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
# ])

# Old dataset class

In [ ]:
# # Here is a basic implementation of the above two TODOs. You can assume the first TODO is completed correctly.
# # Please modify this code to suit you best, as you decide on your preferred model architecture.
# # For example, below here we are padding every video to 1,000 frames. That may or may not be a good idea.


# class VideoDataset(Dataset):
#     """Dataset for loading videos from a folder. Labels from filename prefix."""

#     def __init__(self, video_dir, frame_size=(224, 224), transform=None):
#         self.video_dir = video_dir
#         self.frame_size = frame_size
#         self.transform = transform

#         self.video_files = [
#             f for f in os.listdir(video_dir)
#             if f.endswith(('.mp4', '.avi', '.mov'))
#         ]

#         self.labels = [
#             int(f.split('_')[0]) for f in self.video_files
#         ]

#     def __len__(self):
#         return len(self.video_files)

#     def __getitem__(self, idx):
#         video_path = os.path.join(self.video_dir, self.video_files[idx])
#         frames = self._load_video(video_path)
#         label = self.labels[idx]

#         if self.transform:
#             frames = self.transform(frames)

#         return frames, label

#     def _load_video(self, path, stride = 5, max_frames = 200):
#         cap = cv2.VideoCapture(path)
#         frames = []
#         frame_count = 0

#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break

#             # only add every `stride`'th frame
#             if frame_count % stride == 0:
#                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 frame = cv2.resize(frame, (self.frame_size[1], self.frame_size[0]))
#                 frames.append(frame)

#             frame_count +=1

#             if len(frames) == max_frames:
#                 break

#         cap.release()

#         frames = torch.from_numpy(np.array(frames)).permute(3, 0, 1, 2).float() / 255

#         return frames


# def collate_fn(batch):
#     frames_list, labels = zip(*batch)
#     target_frames = 200  # Updated to your requested 200

#     padded_frames = []
#     for frames in frames_list:
#         # frames shape: (Channels, Time, Height, Width)
#         num_frames = frames.shape[1]
        
#         if num_frames < target_frames:
#             # 1. Grab the last frame: (C, 1, H, W)
#             last_frame = frames[:, -1:, :, :]
            
#             # 2. Calculate how many times to repeat it
#             padding_size = target_frames - num_frames
            
#             # 3. Create the padding by repeating the last frame
#             padding = last_frame.repeat(1, padding_size, 1, 1)
            
#             # 4. Concatenate along the temporal dimension (dim=1)
#             frames = torch.cat([frames, padding], dim=1)
            
#         elif num_frames > target_frames:
#             # Truncate if the video is too long
#             frames = frames[:, :target_frames, :, :]
            
#         padded_frames.append(frames)

#     # Combine list into a single batch tensor: (Batch, C, T, H, W)
#     frames_batch = torch.stack(padded_frames, dim=0)
#     labels_batch = torch.tensor(labels)

#     return frames_batch, labels_batch


# def get_dataloaders(video_dir, batch_size=5, val_split=0.2, collate_fn = None):
#     """Create train and validation dataloaders."""

#     full_dataset = VideoDataset(video_dir)

#     val_size = int(len(full_dataset) * val_split)
#     train_size = len(full_dataset) - val_size

#     train_dataset, val_dataset = random_split(
#         full_dataset,
#         [train_size, val_size],
#         generator=torch.Generator().manual_seed(42)
#     )

#     train_loader = DataLoader(
#         train_dataset,
#         batch_size=batch_size,
#         shuffle=True,
#         num_workers=0,
#         collate_fn=collate_fn
#     )

#     val_loader = DataLoader(
#         val_dataset,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=0,
#         collate_fn=collate_fn
#     )

#     print(f"Train: {len(train_dataset)} videos, Val: {len(val_dataset)} videos\n")

#     return train_loader, val_loader

# Thirteen: go bigger
- **update** back to vanishing gradients problem

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class VideoResNetModel(nn.Module):
    def __init__(self, use_pretrained=True, num_classes = 10):
        super().__init__()
        
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        for param in self.backbone.parameters():
            param.requires_grad = False  # Gradients are no longer calculated/stored

        # Temporal Processor
        # ResNet18 (after Global Average Pool) outputs 512 features per frame.
        # added one more convolutional layer than before
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(1024, 512, kernel_size=3, stride=2, padding=1), # 100 -> 50
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
            nn.Dropout1d(p = .2),
            nn.Conv1d(512, 256, kernel_size=3, stride=2, padding=1), # 100 -> 50
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
            nn.Dropout1d(p = .2),
            nn.Conv1d(256, 128, kernel_size=3, stride=2, padding=1), # 50 -> 25
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
            nn.Dropout1d(p = .2),
            nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 25 -> 13
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Dropout1d(p = .2),
            nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 13 -> 7
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool1d(1) # sqashes to 1 "frame"
        )
        
        # 5. Classifier
        # Output of temporal is [B, 32, 13]. Flattened = 32 * 1 = 32
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 112),
            nn.LeakyReLU(0.1),
            nn.Dropout(p = .2),
            nn.Linear(112, 224),
            nn.LeakyReLU(0.1),
            nn.Dropout(p = .2),
            nn.Linear(224, 112),
            nn.LeakyReLU(0.1),
            nn.Dropout(p = .2),
            nn.Linear(112, num_classes),
        )

    def forward(self, x):
        # x: [B, C, T, H, W]
        B, C, T, H, W = x.shape
        
        # Reshape for Spatial: [B*T, C, H, W]
        x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
        # Pass through ResNet backbone
        # Output is [B*T, 512, 1, 1]
        spatial_features = self.backbone(x_in) 
        
        # Reshape for Temporal: [B, 512, T]
        temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
        # Temporal Conv
        temporal_out = self.temporal_conv(temporal_in)
        
        # Classify
        return self.classifier(temporal_out)

# More model evaluation code

# Continuing learning at a lower learning rate

In [ ]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

# 1. Re-initialize Model, Optimizer, and Scheduler
model = VideoResNetModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Initial LR doesn't matter yet
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min')

# 2. Load the Checkpoint
checkpoint = torch.load('best_video_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# scheduler.load_state_dict(checkpoint['schedular_state_dict'])
left_off_epoch = checkpoint['epoch']
best_val_loss_so_far = checkpoint.get('best_val_loss', float('inf'))

model.to(device)

# We iterate through the param_groups to force a new LR
new_lr = 5e-5 
for param_group in optimizer.param_groups:
    param_group['lr'] = new_lr

print(f"Resuming from Epoch {left_off_epoch}. LR reset to {new_lr}")

Resuming from Epoch 29. LR reset to 5e-05


# PICK UP HERE TOMORROW - CONTINUE TRAINING WITH SLIGHTLY LOWER LEARNING RATE

In [ ]:
# setting a manual seed for reproducibility

print(f"Using device: {device}\n")
model, train_losses, train_acces, val_losses, val_acces = train_model(
    model, 
    epochs = 30, 
    lr = 1e-5, 
    weight_decay = 5e-4, 
    start_epoch = left_off_epoch, 
    best_val_loss = best_val_loss_so_far
    )

Using device: mps

Train: 66 videos, Val: 16 videos

Got dataloaders.

Go time. Let the training commence.

Loss on batch 0: 1.4939
Loss on batch 1: 1.6236
Loss on batch 2: 1.5860
Loss on batch 3: 1.3730
Loss on batch 4: 1.5401
Loss on batch 5: 1.8942
Loss on batch 6: 2.0229
Loss on batch 7: 1.8432
Loss on batch 8: 1.7422
Loss on batch 9: 1.4806
Loss on batch 10: 1.5821
Loss on batch 11: 1.9734
Loss on batch 12: 1.5439
Loss on batch 13: 2.3419
Sample Preds: [4 3 3 9 3]
Sample Preds: [3 3 1 3 4]
Sample Preds: [6 1 6 7 4]
Sample Preds: [1]
Epoch 29/58 | LR: 1.0e-05 | Train Loss: 1.7172, Train Acc: 0.64 | Val Loss: 1.9717, Val Acc: 0.25 | RAM: 0.82GB | Weight Norm: 26.211
Loss on batch 0: 1.7581
Loss on batch 1: 1.9094
Loss on batch 2: 1.8692
Loss on batch 3: 1.5548
Loss on batch 4: 1.3226
Loss on batch 5: 1.6721
Loss on batch 6: 1.9113
Loss on batch 7: 1.8638
Loss on batch 8: 1.6243
Loss on batch 9: 1.8687
Loss on batch 10: 1.9364
Loss on batch 11: 1.5082
Loss on batch 12: 1.7780
Loss on

KeyboardInterrupt: 

<!-- lowering weight decay -->

In [ ]:
# # More comprehensive model evaluation (no transformations)
# full_dataset = VideoDataset('./video-data')

# eval_loader = DataLoader(
#         full_dataset,
#         batch_size= 5,
#         shuffle = False,
#         num_workers = 0,
#         collate_fn = collate_fn
#         )

# eval_loss, eval_acc = evaluate(model_1, eval_loader, nn.MSELoss(), "mps", print_predictions = False)
# print(f"Model 1 --- Val Loss: {eval_loss:.4f}, Val Acc: {eval_acc:.2f}")
# eval_loss, eval_acc = evaluate(model_2, eval_loader, nn.MSELoss(), "mps", print_predictions = False)
# print(f"Model 2 --- Val Loss: {eval_loss:.4f}, Val Acc: {eval_acc:.2f}")

# Evaluate and report results

In [ ]:
current = model
previous_best = VideoResNetModel()
checkpoint = torch.load('best_video_model.pth')
previous_best.load_state_dict(checkpoint['model_state_dict'])
previous_best.to("mps")
current.to("mps")
_, _, _ = run_inference(current)
_, _, _ = run_inference(previous_best)

Using device: mps

Running inference on 77 test videos...


✓  pred=2  true=2  |  2_sadfasjldkfjaseifj.mp4
✗  pred=3  true=2  |  2_sdafkjaslkclaksdjkas.mp4
✓  pred=4  true=4  |  4_kling_20251206_Text_to_Video_Generate_a_28_0.mp4
✓  pred=3  true=3  |  3_kling_dskfseu.mp4
✓  pred=4  true=4  |  4_kling_20251209_Text_to_Video_Generate_a_190_0.mp4
✓  pred=3  true=3  |  3_kling_kdjflaskdjf.mp4
✗  pred=4  true=2  |  2_dsalkfjalwkenlke.mp4
✓  pred=3  true=3  |  3_dsjlaeijlksjdfie.mp4
✓  pred=3  true=3  |  3_kling_20251205_Text_to_Video_On_a_playg_5028_0.mp4
✓  pred=4  true=4  |  4_sadlfkjlknewkjejk.mp4
✓  pred=3  true=3  |  3_kling_20251206_Text_to_Video_Generate_a_315_2.mp4
✓  pred=4  true=4  |  4_kling_20251209_Text_to_Video_Generate_a_561_1.mp4
✓  pred=2  true=2  |  2_difficult_2.mp4
✗  pred=4  true=3  |  3_sdlkjslndflkseijlkjef.mp4
✓  pred=3  true=3  |  3_kling_20251206_Text_to_Video_Generate_a_315_0.mp4
✗  pred=4  true=6  |  6_dfjewaijsldkjfsaef.mp4
✓  pred=4  true=4  |  4_kling_20251209_

In [ ]:
# More comprehensive model evaluation (no transformations)
full_dataset = VideoDataset('./video-data')

eval_loader = DataLoader(
        full_dataset,
        batch_size= 5,
        shuffle = False,
        num_workers = 0,
        collate_fn = collate_fn
        )

model.to("mps")
eval_loss, eval_acc = evaluate(model, eval_loader, nn.CrossEntropyLoss(), "mps", print_predictions = True)
print(f"Val Loss: {eval_loss:.4f}, Val Acc: {eval_acc:.2f}")

NameError: name 'VideoDataset' is not defined

# trying to get a sense of what my augmented data looks like

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# def visualize_transformed_video(video_tensor, num_frames=5):
#     """
#     video_tensor: Tensor of shape (C, T, H, W)
#     num_frames: How many frames from the sequence to display
#     """
#     # 1. Move to CPU and Permute to (T, H, W, C) for plotting
#     # [C, T, H, W] -> [T, H, W, C]
#     video = video_tensor.permute(1, 2, 3, 0).cpu().numpy()
    
#     # 2. Denormalize
#     # These are the ImageNet constants you used in your transform
#     mean = np.array([0.485, 0.456, 0.406])
#     std = np.array([0.229, 0.224, 0.225])
#     video = (video * std) + mean
#     # video = np.clip(video, 0, 1) # Ensure values are within [0, 1]
    
#     # 3. Select frames to show (evenly spaced)
#     total_frames = video.shape[0]
#     indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
#     # 4. Plot
#     fig, axes = plt.subplots(1, num_frames, figsize=(20, 5))
#     for i, idx in enumerate(indices):
#         axes[i].imshow(video[idx])
#         axes[i].set_title(f"Frame {idx}")
#         axes[i].axis('off')
#     plt.show()

In [ ]:
# # Grab one batch from your training loader
# frames_batch, labels_batch = next(iter(val_loader))

# print(f"Viewing video with label: {labels_batch[0]}")
# visualize_transformed_video(frames_batch[0], num_frames=10)
# visualize_transformed_video(frames_batch[1], num_frames=10)
# visualize_transformed_video(frames_batch[2], num_frames=10)
# visualize_transformed_video(frames_batch[3], num_frames=10)
# visualize_transformed_video(frames_batch[4], num_frames=10)

# Twelfth attempt
- trying to make the dataset larger and acheiving class balance in the training data:
 - take all videos with 2, cut them in half, save them down with label 1
 - take all videos with 3-5 pushups, stitch temporally with itself, label 6, 8, and 10.
 - take all videos with 4, stitch 75% of frames at the end, label with 7
 - take all videos with 3, stitch twice against itself, label with 9.


When stitching videos, we do "palindrome stitching". i.e. stitch a copy of the video _in reverse_. That way there's no jump cut at the moment of the stitch, which may confuse the model as it tracks the movement of the person. 

In [ ]:
# import cv2
# video_paths = [f for f in os.listdir("./video-data") if f.endswith(('.mp4', '.avi', '.mov'))]

### Checking how many frames my original videos are
 - they are from 96 to 325 frames long
 - uncomment the below chunk to see

In [ ]:
# folder_path = './video-data'
# frame_count = []
# for video in video_paths:
#     path = os.path.join(folder_path, video)
#     cap = cv2.VideoCapture(path)
#     count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     frame_count.append(count)
#     cap.release()

# print(sorted(set(frame_count)))

### Checking how many of each label we have
- we have way more 2s, 3s, and 4s than anything else. The goal should be to use these as inputs and spread the data over more classes.

In [ ]:
# video_paths = [f for f in os.listdir("./video-data") if f.endswith(('.mp4', '.avi', '.mov'))]
# labels = [int(f.split('_')[0]) for f in video_paths]
# vals, counts = np.unique(labels, return_counts=True)
# np.array([vals, counts])

Some generic functions for extending or cutting the length of some videos

In [ ]:
# def double_length(paths, search_dir = "video-data", out_dir = "train-data"):
    
#     for video_path in paths:
    
#         full_path = f"{search_dir}/{video_path}"

#         original_label = int(video_path.split('_')[0])
#         pattern = f"{original_label}_"
#         rest = video_path.replace(pattern, "", 1)
#         new_label = original_label * 2
        
#         output_path = f"{out_dir}/{new_label}_STITCHED_{rest}"

#         cap = cv2.VideoCapture(full_path)
#         width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps    = cap.get(cv2.CAP_PROP_FPS)
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')

#         out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#         frames = []

#         # 1. Read all frames into memory
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             frames.append(frame)

#         # 2. Write frames Forward (Original)
#         for f in frames:
#             out.write(f)

#         # 3. Write frames Backward (Reversed)
#         # [::-1] reverses the list; we skip the very last frame to avoid a double-frame pause
#         for f in frames[::-1]:
#             out.write(f)

#         cap.release()
#         out.release()
#         print(f"{output_path}: Done! Created a sequence of {len(frames) * 2} frames.")


# def triple_length(paths, search_dir = "video-data", out_dir = "train-data"):
    
#     for video_path in paths:
    
#         full_path = f"{search_dir}/{video_path}"

#         original_label = int(video_path.split('_')[0])
#         pattern = f"{original_label}_"
#         rest = video_path.replace(pattern, "", 1)
#         new_label = original_label * 3
        
#         output_path = f"{out_dir}/{new_label}_STITCHED_{rest}"

#         cap = cv2.VideoCapture(full_path)
#         width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps    = cap.get(cv2.CAP_PROP_FPS)
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')

#         out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#         frames = []

#         # 1. Read all frames into memory
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             frames.append(frame)

#         # 2. Write frames Forward (Original)
#         for f in frames:
#             out.write(f)

#         # 3. Write frames Backward (Reversed)
#         # [::-1] reverses the list; we skip the very last frame to avoid a double-frame pause
#         for f in frames[::-1]:
#             out.write(f)

#         # write frames forward again
#         for f in frames:
#             out.write(f)        

#         cap.release()
#         out.release()
#         print(f"{output_path}: Done! Created a sequence of {len(frames) * 3} frames.")

# def three_to_five(paths, search_dir = "video-data", out_dir = "train-data"):

#     for video_path in paths:
    
#         full_path = f"{search_dir}/{video_path}"

#         original_label = 3
#         pattern = f"{original_label}_"
#         rest = video_path.replace(pattern, "", 1)
#         new_label = 5
        
#         output_path = f"{out_dir}/{new_label}_STITCHED_{rest}"

#         cap = cv2.VideoCapture(full_path)
#         width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')

#         out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#         # 1. Load all frames into a list
#         frames = []
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             frames.append(frame)

#         # 2. Define the cut-off for the palindrome
#         # To add 2 pushups to a 3-pushup video, we need 60% of the frames
#         # went with 66 to account for some lag before the pushup begins
#         cutoff_index = int(len(frames) * 0.66)

#         # 3. Write the Original 4 Pushups (Forward)
#         for f in frames:
#             out.write(f)

#         # 4. Write the Palindrome (Reverse)
#         # We start from the very end and go backwards until we hit the 75% mark
#         # This adds the 3 pushups in reverse
#         palindrome_frames = frames[::-1] # Reverse the full list
#         clipped_palindrome = palindrome_frames[:cutoff_index]

#         for f in clipped_palindrome:
#             out.write(f)

#         cap.release()
#         out.release()

#         print(f"{output_path}: Total frames in new video = {len(frames) + len(clipped_palindrome)}")



# def four_to_seven(paths, search_dir = "video-data", out_dir = "train-data"):

#     for video_path in paths:
    
#         full_path = f"{search_dir}/{video_path}"

#         original_label = 4
#         pattern = f"{original_label}_"
#         rest = video_path.replace(pattern, "", 1)
#         new_label = 7
        
#         output_path = f"{out_dir}/{new_label}_STITCHED_{rest}"

#         cap = cv2.VideoCapture(full_path)
#         width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')

#         out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#         # 1. Load all frames into a list
#         frames = []
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             frames.append(frame)

#         # 2. Define the cut-off for the palindrome
#         # To add 3 pushups to a 4-pushup video, we need 75% of the frames
#         # went with 80% to account for dead time before first pushup
#         cutoff_index = int(len(frames) * 0.80)

#         # 3. Write the Original 4 Pushups (Forward)
#         for f in frames:
#             out.write(f)

#         # 4. Write the Palindrome (Reverse)
#         # We start from the very end and go backwards until we hit the 75% mark
#         # This adds the 3 pushups in reverse
#         palindrome_frames = frames[::-1] # Reverse the full list
#         clipped_palindrome = palindrome_frames[:cutoff_index]

#         for f in clipped_palindrome:
#             out.write(f)

#         cap.release()
#         out.release()

#         print(f"{output_path}: Total frames in new video = {len(frames) + len(clipped_palindrome)}")


# def cut_in_half(paths, search_dir = "video-data", out_dir = "train-data"):

#     for video_path in paths:
        
#         full_path = f"{search_dir}/{video_path}"

#         original_label = int(video_path.split('_')[0])
#         pattern = f"{original_label}_"
#         rest = video_path.replace(pattern, "", 1)
#         new_label = int(original_label / 2)
        
#         cap = cv2.VideoCapture(full_path)

#         # Get video properties
#         width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#         fps    = cap.get(cv2.CAP_PROP_FPS)
#         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')

#         # Calculate the midpoint
#         midpoint = total_frames // 2

#         def save_segment(start_frame, end_frame, output_name):
#             out = cv2.VideoWriter(output_name, fourcc, fps, (width, height))
#             cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
            
#             for i in range(start_frame, end_frame):
#                 ret, frame = cap.read()
#                 if not ret:
#                     break
#                 out.write(frame)
#             out.release()

#         output_path_1 = f"{out_dir}/{new_label}_SPLIT_FIRST_{rest}"
#         output_path_2 = f"{out_dir}/{new_label}_SPLIT_SECOND_{rest}"

#         # Save both halves
#         save_segment(0, midpoint, output_path_1)
#         save_segment(midpoint, total_frames, output_path_2)

#         cap.release()
#         print(f"Video split successfully at frame {midpoint}!")

### Apply our operations to our input videos
- I grab random samples of videos each with 2, 3 and 4 pushups, and I selectively cut and copies together so as to create more videos containing counts of 1, 5, 6, 7, 8, 9, and 10.

In [ ]:
# video_paths = [f for f in os.listdir("./video-data") if f.endswith(('.mp4', '.avi', '.mov'))]
# all_paths = [s for s in video_paths if s.startswith(('2', '3', '4'))]

# # grab some for operations
# p = .6
# k = int(len(all_paths) * p)
# random.seed(42)
# paths_for_use = random.sample(all_paths, k)

# # lets
# # - half all the 2s, (to get 1s)
# # - triple all the 3s (to get 9s)
# # - double all the 4s (to get 8s)
# twos = [s for s in paths_for_use if s.startswith(('2'))]
# threes = [s for s in paths_for_use if s.startswith(('3'))]
# fours = [s for s in paths_for_use if s.startswith(('4'))]

# # hold out some 3s and 4s to make 5s and 7s, and some 2s to make 6s
# p = .5
# two_k = int(len(twos) * p)
# three_k = int(len(threes) * p)
# four_k = int(len(fours) * p)
# random.seed(42)
# twos_for_half = random.sample(twos, two_k)
# twos_for_trip = [v for v in twos if v not in twos_for_half]
# threes_for_trip = random.sample(threes, three_k)
# threes_for_fives = [v for v in threes if v not in threes_for_trip]
# fours_for_dub = random.sample(fours, four_k)
# fours_for_sevens = [v for v in fours if v not in fours_for_dub]

# # refresh the new training data folder
# folder_to_delete = './train-data'
# if os.path.exists(folder_to_delete):
#     shutil.rmtree(folder_to_delete)
#     print(f"Removed {folder_to_delete}")
# else:
#     print("Folder not found.")
# os.makedirs("./train-data", exist_ok=True)

# # make new videos and save to "./train-data" (the default output directory for the function)
# cut_in_half(twos_for_half)
# triple_length(twos_for_trip)
# triple_length(threes_for_trip)
# three_to_five(threes_for_fives)
# double_length(fours_for_dub)
# four_to_seven(fours_for_sevens)

# # now, grab half of the 5s we just made, and turn them into 10s
# new_vids = [f for f in os.listdir("./train-data") if f.endswith(('.mp4', '.avi', '.mov'))]
# fives = [s for s in new_vids if s.startswith(('5'))]
# p = .5
# five_k = int(len(fives) * p)
# random.seed(42)
# fives_for_dup = random.sample(fives, five_k)
# double_length(fives_for_dup, search_dir = "train-data", out_dir = "train-data")

# # delete the original fives we used
# for filename in fives_for_dup:
#     file_path = os.path.join("train-data", filename)
#     # Check if file exists to avoid an error
#     if os.path.exists(file_path):
#         os.remove(file_path)
#         print(f"Deleted: {filename}")
#     else:
#         print(f"File not found: {filename}")

# # copy held out videos from "video-data" to "train-data"
# remaining = [v for v in video_paths if v not in paths_for_use]
# for file_name in remaining:
#     source_path = os.path.join('./video-data', file_name)
#     dest_path = os.path.join('./train-data', file_name)
#     shutil.copy2(source_path, dest_path)
#     print(f"Copied {file_name} to './train-data'")

# Trying to instead save copies of all videos flipped horizontally and played backwards

To take a video, flip it, and reverse, we naturally MUST write a function called "missy_elliot"

In [ ]:
def missy_elliot(paths, search_dir = "video-data", out_dir = "train-data"):
    
    for video_path in paths:
    
        full_path = f"{search_dir}/{video_path}"

        original_label = int(video_path.split('_')[0])
        pattern = f"{original_label}_"
        rest = video_path.replace(pattern, "", 1)
        new_label = original_label
        
        output_path = f"{out_dir}/{new_label}_MISSYED_{rest}"

        cap = cv2.VideoCapture(full_path)
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps    = cap.get(cv2.CAP_PROP_FPS)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')

        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        frames = []

        # 1. Read frames and FLIP them horizontally
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # cv2.flip(frame, 1) flips horizontally
            flipped_frame = cv2.flip(frame, 1)
            frames.append(flipped_frame)

        # 3. Write frames Backward (Reversed)
        # [::-1] reverses the list; we skip the very last frame to avoid a double-frame pause
        for f in frames[::-1]:
            out.write(f)

        cap.release()
        out.release()
        print(f"{output_path}: Done! Created a missy sequence of {len(frames) * 2} frames.")

In [ ]:
video_paths = [f for f in os.listdir("./video-data") if f.endswith(('.mp4', '.avi', '.mov'))]

# refresh the new training data folder
folder_to_delete = './train-data'
if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Removed {folder_to_delete}")
else:
    print("Folder not found.")
os.makedirs("./train-data", exist_ok=True)

# make new videos and save to "./train-data" (the default output directory for the function)
missy_elliot(video_paths)

# copy  "video-data" to "train-data"
for file_name in video_paths:
    source_path = os.path.join('./video-data', file_name)
    dest_path = os.path.join('./train-data', file_name)
    shutil.copy2(source_path, dest_path)
    print(f"Copied {file_name} to './train-data'")

Removed ./train-data
train-data/2_MISSYED_sadfasjldkfjaseifj.mp4: Done! Created a missy sequence of 318 frames.
train-data/2_MISSYED_sdafkjaslkclaksdjkas.mp4: Done! Created a missy sequence of 168 frames.
train-data/4_MISSYED_kling_20251206_Text_to_Video_Generate_a_28_0.mp4: Done! Created a missy sequence of 482 frames.
train-data/3_MISSYED_kling_dskfseu.mp4: Done! Created a missy sequence of 456 frames.
train-data/4_MISSYED_kling_20251209_Text_to_Video_Generate_a_190_0.mp4: Done! Created a missy sequence of 482 frames.
train-data/3_MISSYED_kling_kdjflaskdjf.mp4: Done! Created a missy sequence of 422 frames.
train-data/2_MISSYED_dsalkfjalwkenlke.mp4: Done! Created a missy sequence of 312 frames.
train-data/3_MISSYED_dsjlaeijlksjdfie.mp4: Done! Created a missy sequence of 298 frames.
train-data/3_MISSYED_kling_20251205_Text_to_Video_On_a_playg_5028_0.mp4: Done! Created a missy sequence of 384 frames.
train-data/4_MISSYED_sadlfkjlknewkjejk.mp4: Done! Created a missy sequence of 426 frame

### Recheck label distribution
- **OLD** The labels are now more evenly distributed across all the classes we're trying to predict
- **NEW** we now have more training videos

In [ ]:
video_paths = [f for f in os.listdir("./train-data") if f.endswith(('.mp4', '.avi', '.mov'))]
labels = [int(f.split('_')[0]) for f in video_paths]
vals, counts = np.unique(labels, return_counts=True)
np.array([vals, counts])

array([[ 1,  2,  3,  4,  5,  6,  7],
       [ 2, 32, 68, 42,  4,  4,  2]])

This model was my eleventh attempt. 

In a previous version, we achieved significant overfitting (94% train acc, 75% test acc)

So, I made the following changes:
- adding small dropouts to both final classifier head and temporal layers
- adding adaptive max pooling to end of temporal layer

In a slightly different version of this attempt, I added back in some light data augmentation, but it made the model way worse

In [ ]:
# import torch
# import torch.nn as nn
# from torchvision import models

# class VideoResNetModel(nn.Module):
#     def __init__(self, use_pretrained=True, num_classes = 10):
#         super().__init__()
        
#         resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if use_pretrained else None)
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         for param in self.backbone.parameters():
#             param.requires_grad = False  # Gradients are no longer calculated/stored

#         # Temporal Processor
#         # ResNet18 (after Global Average Pool) outputs 512 features per frame.
#         self.temporal_conv = nn.Sequential(
#             nn.Conv1d(512, 256, kernel_size=3, stride=2, padding=1), # 100 -> 50
#             nn.BatchNorm1d(256),
#             nn.LeakyReLU(0.1), # LeakyReLU helps with vanishing gradients
#             nn.Dropout1d(p = .2),
#             nn.Conv1d(256, 128, kernel_size=3, stride=2, padding=1), # 50 -> 25
#             nn.BatchNorm1d(128),
#             nn.LeakyReLU(0.1),
#             nn.Dropout1d(p = .2),
#             nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1), # 25 -> 13
#             nn.BatchNorm1d(64),
#             nn.LeakyReLU(0.1),
#             nn.Dropout1d(p = .2),
#             nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1),  # 13 -> 7
#             nn.BatchNorm1d(32),
#             nn.LeakyReLU(0.1),
#             nn.AdaptiveAvgPool1d(1) # sqashes to 1 "frame"
#         )
        
#         # output of temporal is [B, 32, 1]. Flattened = 32 * 1 = 32
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(32, 112),
#             nn.LeakyReLU(0.1),
#             nn.Dropout(p = .2),
#             nn.Linear(112, num_classes),
#         )

#     def forward(self, x):
        
#         # For each video, ...
        
#         # ... take the shape, x: [B, C, T, H, W],
#         B, C, T, H, W = x.shape
        
#         # ... reshape for Spatial: [B*T, C, H, W], 
#         x_in = x.transpose(1, 2).reshape(B * T, C, H, W) 
        
#         # ... pass through ResNet backbone (output is [B*T, 512, 1, 1]),
#         spatial_features = self.backbone(x_in) 
        
#         # ... reshape for Temporal: [B, 512, T], 
#         temporal_in = spatial_features.view(B, T, -1).transpose(1, 2)
        
#         # ... pass through temporal layers,
#         temporal_out = self.temporal_conv(temporal_in)
        
#         # ... and classify.
#         return self.classifier(temporal_out)